# Olist E-Commerce Data Analysis
_________________________________
## Objective
This project analyzes the Brazilian e-commerce dataset to understand:
- customer satisfaction drivers
- delivery performance
- payment behavior
- product category trends

## Approach
- Data cleaning and preparation
- Building an order-level dataset
- Exploratory data analysis
- Hypothesis testing (A/B testing)

  Importing Libraries

In [9]:
from scipy.stats import ttest_ind, chi2_contingency, f_oneway

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

  Loading .csv files

In [10]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
drive.mount('/content/drive')

orders = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_orders_dataset.csv')
customers = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_customers_dataset.csv')
items = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_order_payments_dataset.csv') # Corrected file path
reviews = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/Python/Olist/olist_sellers_dataset.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


________________
# Data Cleaning
________________
Each table was cleaned individually to:
- standardize formats
- handle missing values
- ensure consistency before merging

Key decisions:
- dates converted using safe parsing
- categorical columns standardized
- no aggressive row dropping at this stage

  Initial Data Understanding

In [11]:
tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers
}

for name, df in tables.items():
    print("=" * 70)
    print(f"Table: {name}")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nDtypes:")
    print(df.dtypes)
    print("\nTop 10 null counts:")
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    print()

Table: orders
Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Top 10 null counts:
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

Table: customers
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'c

# Data Cleaning | Table by Table

Orders Table Cleaning

In [46]:
order_date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors = "coerce")

display(orders.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


  Customers Table Cleaning

In [47]:
customers['customer_city'] = customers['customer_city'].astype(str).str.strip().str.lower()
customers['customer_state'] = customers['customer_state'].astype(str).str.strip().str.upper()

display(customers.head())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


  Items Table Cleaning

In [48]:
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors = "coerce")
items['price'] = pd.to_numeric(items['price'], errors = "coerce")
items['freight_value'] = pd.to_numeric(items['freight_value'], errors = "coerce")

display(items.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


  Payments Table Cleaning

In [49]:
payments['payment_type'] = payments['payment_type'].apply(lambda x: str(x).strip().lower())
payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors = "coerce")
payments['payment_installments'] = pd.to_numeric(payments['payment_installments'], errors = "coerce")

display(payments.head())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


  Reviews Table Cleaning

In [50]:
review_date_cols = [
    'review_creation_date',
    'review_answer_timestamp'
]
for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors = "coerce")

reviews['review_score'] = pd.to_numeric(reviews['review_score'], errors = "coerce")

display(reviews.head())

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


  Products Table Cleaning

In [51]:
products['product_category_name'] = products['product_category_name'].astype(str).str.strip().str.lower()

product_num_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

display(products.head())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


  Sellers Table Cleaning

In [52]:
sellers['seller_city'] = sellers['seller_city'].astype(str).str.strip().str.lower()
sellers['seller_state'] = sellers['seller_state'].astype(str).str.strip().str.upper()

display(sellers.head())

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


  Checking Duplicates in Base Tables

In [19]:
print("orders duplicate order_id:", orders["order_id"].duplicated().sum())
print("customers duplicate customer_id:", customers["customer_id"].duplicated().sum())
print("products duplicate product_id:", products["product_id"].duplicated().sum())
print("sellers duplicate seller_id:", sellers["seller_id"].duplicated().sum())
print("reviews duplicate review_id:", reviews["review_id"].duplicated().sum())

orders duplicate order_id: 0
customers duplicate customer_id: 0
products duplicate product_id: 0
sellers duplicate seller_id: 0
reviews duplicate review_id: 814


  Understanding One-to-Many Relationships

In [20]:
print("Average items per order:", items.groupby('order_id').size().mean())
print("Average payments per order:", payments.groupby('order_id').size().mean())
print("Average reviews per order:", reviews.groupby('order_id').size().mean())

Average items per order: 1.1417306873695092
Average payments per order: 1.0447103781174578
Average reviews per order: 1.0055841010205426


# Creating Order-Level Aggregated Tables

Items Table Aggregation

In [53]:
items_agg = (
    items.groupby('order_id')
    .agg(
        item_count = ("order_item_id", "count"),
        total_price = ("price", "sum"),
        total_freight = ("freight_value", "sum"),
        unique_seller_count = ("seller_id", "nunique"),
        unique_product_count = ("product_id", "nunique")
    )
    .reset_index()
)

items_agg["order_value"] = items_agg["total_price"] + items_agg["total_freight"]

display(items_agg.head())

,order_id,item_count,total_price,total_freight,unique_seller_count,unique_product_count,order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1,218.04


  Payment Table Aggregation

In [54]:
def safe_mode(series):
  mode_vals =series.mode()
  return mode_vals.iloc[0] if not mode_vals.empty else np.nan

payments_agg = (
    payments.groupby("order_id")
    .agg(
        payment_count = ("payment_sequential", "count"),
        total_payment_value = ("payment_value", "sum"),
        avg_installments = ("payment_installments", "mean"),
        main_payment_type = ("payment_type", safe_mode)
    )
    .reset_index()
)

payments_agg.head()

,order_id,payment_count,total_payment_value,avg_installments,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2.0,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3.0,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5.0,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2.0,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3.0,credit_card


Reviews Table Aggregation

In [55]:
reviews_agg = (
    reviews.groupby("order_id")
    .agg(
        review_score = ("review_score", "mean")
    )
    .reset_index()
)

reviews_agg.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


  Category Table Aggregation

In [56]:
items_products = items.merge(products [["product_id","product_category_name"]], on = "product_id", how="left")

category_agg = (
    items_products.groupby("order_id")["product_category_name"]
    .agg(safe_mode)
    .reset_index()
    .rename(columns={"product_category_name": "main_category"})
)

category_agg.head()

,order_id,main_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,ferramentas_jardim


# Building Master Order-Level Dataset

## Data Modeling Approach

Due to one-to-many relationships (items, payments),
direct joins would create duplicate rows.

To solve this:
- item-level and payment-level tables were aggregated
- a clean order-level dataset was created

This ensured consistent analysis.

In [57]:
master_df = (
    orders
    .merge(customers, on = "customer_id", how = "left")
    .merge(items_agg, on = "order_id", how = "left")
    .merge(payments_agg, on = "order_id", how = "left")
    .merge(reviews_agg, on = "order_id", how = "left")
    .merge(category_agg, on = "order_id", how = "left")
)

print(master_df.shape)
master_df.head()

(99441, 24)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,total_price,total_freight,unique_seller_count,unique_product_count,order_value,payment_count,total_payment_value,avg_installments,main_payment_type,review_score,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,1.0,1.0,38.71,3.0,38.71,1.0,voucher,4.0,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,1.0,1.0,141.46,1.0,141.46,1.0,boleto,4.0,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,1.0,1.0,179.12,1.0,179.12,3.0,credit_card,5.0,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,1.0,1.0,72.20,1.0,72.20,1.0,credit_card,5.0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,1.0,1.0,28.62,1.0,28.62,1.0,credit_card,5.0,papelaria


#  Feature Engineering
_________________________
New variables were created to support analysis:

- order_value → total purchase amount
- delivery_time_days → delivery duration
- is_delayed → delay indicator
- is_satisfied → customer satisfaction proxy

In [58]:
#Delivery Time
master_df["delivery_time_days"] = (master_df["order_delivered_customer_date"] - master_df["order_purchase_timestamp"]).dt.days

#Estimated vs Actual Delay
master_df["delivery_delay_days"] = (master_df["order_delivered_customer_date"] - master_df["order_estimated_delivery_date"]).dt.days

#Is it delayed?
master_df["is_delayed"] = np.where(master_df["delivery_delay_days"] > 0, 1, 0)

#Order Status Flags
master_df["is_delivered"] = np.where(master_df["order_status"] == "delivered", 1, 0)
master_df["is_cancelled"] = np.where(master_df["order_status"] == "cancelled", 1, 0)

#Satisfaction Flag
master_df["is_satisfied"] = np.where(master_df["review_score"] >= 4, 1, 0)

#Time Features
master_df["purchase_year"] = master_df["order_purchase_timestamp"].dt.year
master_df["purchase_month"] = master_df["order_purchase_timestamp"].dt.month
master_df["purchase_weekday"] = master_df["order_purchase_timestamp"].dt.weekday

display(master_df.head())
print("")
print("=" * 40)
print("")
print(master_df.isnull().sum().sort_values(ascending = False).head(20))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,total_price,total_freight,unique_seller_count,unique_product_count,order_value,payment_count,total_payment_value,avg_installments,main_payment_type,review_score,main_category,delivery_time_days,delivery_delay_days,is_delayed,is_delivered,is_cancelled,is_satisfied,purchase_year,purchase_month,purchase_weekday
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,1.0,1.0,38.71,3.0,38.71,1.0,voucher,4.0,utilidades_domesticas,8.0,-8.0,0,1,0,1,2017,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,1.0,1.0,141.46,1.0,141.46,1.0,boleto,4.0,perfumaria,13.0,-6.0,0,1,0,1,2018,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,1.0,1.0,179.12,1.0,179.12,3.0,credit_card,5.0,automotivo,9.0,-18.0,0,1,0,1,2018,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,1.0,1.0,72.20,1.0,72.20,1.0,credit_card,5.0,pet_shop,13.0,-13.0,0,1,0,1,2017,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,1.0,1.0,28.62,1.0,28.62,1.0,credit_card,5.0,papelaria,2.0,-10.0,0,1,0,1,2018,2,1




order_delivered_customer_date    2965
delivery_delay_days              2965
delivery_time_days               2965
order_delivered_carrier_date     1783
order_value                       775
total_price                       775
total_freight                     775
unique_seller_count               775
unique_product_count              775
item_count                        775
main_category                     775
review_score                      768
order_approved_at                 160
main_payment_type                   1
total_payment_value                 1
avg_installments                    1
payment_count                       1
order_id                            0
customer_zip_code_prefix            0
customer_state                      0
dtype: int64


#  Building Delivered Data Frame

In [59]:
delivered_df = master_df[master_df["is_delivered"] == 1]. copy()

#Clearing NaN columns
delivered_df = delivered_df.dropna(subset = [
    "delivery_time_days",
    "review_score",
    "order_value"
])

print("master_df shape:", master_df.shape)
print("delivered_df shape:", delivered_df.shape)

master_df shape: (99441, 33)
delivered_df shape: (95824, 33)


# Descriptive Statistics

In [28]:
#Order Value Statistics
order_value_stats = delivered_df["order_value"].agg(["mean", "median", "std", "min", "max"])
print("Order Value Statistics")
print(order_value_stats)
print("")
print("=" * 35)
print("")

#Delivery Duration Statistics
delivery_stats = delivered_df["delivery_time_days"].agg(["mean", "median", "std", "min", "max"])
print("Delivery Duration Statistics")
print(delivery_stats)
print("")
print("=" * 35)
print("")

#Review Score Statistics
review_stats = delivered_df["review_score"].agg(["mean", "median", "std", "min", "max"])
print("Review Score Statistics")
print(review_stats)
print("")
print("=" * 35)
print("")

#Mod Example
print("Main payment type mode:", delivered_df["main_payment_type"].mode())
print("Main category mode:", delivered_df["main_category"].mode())

Order Value Statistics
mean        159.563099
median      105.280000
std         217.494171
min           9.590000
max       13664.080000
Name: order_value, dtype: float64


Delivery Duration Statistics
mean       12.052273
median     10.000000
std         9.466046
min         0.000000
max       208.000000
Name: delivery_time_days, dtype: float64


Review Score Statistics
mean      4.156158
median    5.000000
std       1.283615
min       1.000000
max       5.000000
Name: review_score, dtype: float64


Main payment type mode: 0    credit_card
Name: main_payment_type, dtype: object
Main category mode: 0    cama_mesa_banho
Name: main_category, dtype: object


________________
# EDA | Core Analysis
__________________
### Insight: Order Value Distribution

- Majority of orders are concentrated in lower price ranges
- A small number of high-value orders create a long tail

Business implication:
Pricing strategy and product mix may influence revenue concentration

In [29]:
#Order value distribution
display(delivered_df["order_value"].describe())

,order_value
count,95824.000000
mean,159.563099
std,217.494171
min,9.590000
25%,61.800000
50%,105.280000
75%,176.160000
max,13664.080000


In [30]:
#Main Category Distribution
category_counts = (
    delivered_df["main_category"]
    .value_counts()
    .head(10)
    .reset_index()
)

category_counts.columns = ["main_category", "order_count"]

print("Top 10 main categories:")
display(category_counts)

Top 10 main categories:


,main_category,order_count
0,cama_mesa_banho,9132
1,beleza_saude,8590
2,esporte_lazer,7448
3,informatica_acessorios,6469
4,moveis_decoracao,6136
5,utilidades_domesticas,5605
6,relogios_presentes,5416
7,telefonia,4045
8,automotivo,3785
9,brinquedos,3753


In [60]:
#Payment type distribution
payment_counts = (
    delivered_df["main_payment_type"]
    .value_counts()
    .reset_index()
)

payment_counts.columns = ["payment_type", "count"]
display(payment_counts)

,payment_type,count
0,credit_card,73439
1,boleto,19062
2,voucher,1845
3,debit_card,1477


In [32]:
#Monthly order trend
monthly_orders = (
    delivered_df.groupby("purchase_month")
    .agg(
        order_count = ("order_id", "nunique"),
        avg_order_value = ("order_value", "mean"),
        avg_review_score = ("review_score", "mean")
    )
    .reset_index()
)

monthly_orders

,purchase_month,order_count,avg_order_value,avg_review_score
0,1,7754,153.906059,4.117359
1,2,8150,150.733836,3.946421
2,3,9475,160.124256,3.913034
3,4,9042,167.465714,4.188841
4,5,10239,164.502687,4.236937
5,6,9183,162.882832,4.278722
6,7,9960,158.161728,4.294863
7,8,10495,154.404882,4.310815
8,9,4119,169.160187,4.266691
9,10,4708,167.726317,4.194456


### Insight: Delivery Time vs Customer Satisfaction

- Longer delivery times are associated with lower review scores

Business implication:
Delivery performance is a key driver of customer satisfaction

In [33]:
#Delivery vs Review
delivery_review = (
    delivered_df.groupby("delivery_time_days")
    .agg(avg_review_score = ("review_score", "mean"),
         order_count = ("order_id", "nunique"))
    .reset_index()
)

display(delivery_review.head())


,delivery_time_days,avg_review_score,order_count
0,0.0,4.230769,13
1,1.0,4.494904,1570
2,2.0,4.477937,3150
3,3.0,4.429628,3832
4,4.0,4.426596,4809


In [34]:
#Correlation Analysis
corr_cols = [
    "order_value",
    "delivery_time_days",
    "delivery_delay_days",
    "review_score",
    "item_count",
    "payment_count",
    "avg_installments"
]

corr_df = delivered_df[corr_cols].corr()
display(corr_df)

,order_value,delivery_time_days,delivery_delay_days,review_score,item_count,payment_count,avg_installments
order_value,1.000000,0.069324,-0.018974,-0.042095,0.189506,0.000983,0.318392
delivery_time_days,0.069324,1.000000,0.601832,-0.334105,-0.019689,0.003746,0.051252
delivery_delay_days,-0.018974,0.601832,1.000000,-0.267255,-0.031922,-0.004676,-0.031482
review_score,-0.042095,-0.334105,-0.267255,1.000000,-0.123428,-0.002924,-0.030715
item_count,0.189506,-0.019689,-0.031922,-0.123428,1.000000,-0.002228,0.067802
payment_count,0.000983,0.003746,-0.004676,-0.002924,-0.002228,1.000000,-0.063417
avg_installments,0.318392,0.051252,-0.031482,-0.030715,0.067802,-0.063417,1.000000


_________________________
# Visualizations
__________________________

In [35]:
import plotly.express as px
import plotly.graph_objects as go

In [36]:
#Order Value Histogram
fig = px.histogram(
    delivered_df,
    x = "order_value",
    nbins = 50,
    title = "Distribution of Order Values",
    labels = {"order_value": "Order Value"}
)

fig.show()

In [37]:
#Order value boxplot
fig = px.box(
    delivered_df,
    y = "order_value",
    title = "Boxplot of Order Values",
)

fig.show()

In [38]:
#Top categories bar chart
fig = px.bar(
    category_counts,
    x = "main_category",
    y = "order_count",
    title = "Top 10 Categories by Order Count"
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

In [61]:
#Payment type pie chart
fig = px.pie(
    payment_counts,
    names = "payment_type",
    values = "count",
    title = "Payment Method Distribution"
)

fig.show()

In [40]:
#Delivery time histogram
fig = px.histogram(
    delivered_df,
    x = "delivery_time_days",
    nbins = 40,
    title = "Distribution of Delivery Times",
    labels = {"delivery_time_days": "Delivery Time (Days)"}
)

fig.show()

In [41]:
#Review score distribution
fig = px.histogram(
    delivered_df,
    x="review_score",
    nbins=5,
    title="Review Score Distribution"
)
fig.show()

In [42]:
#Monthly order trend line
fig = px.line(
    monthly_orders,
    x="purchase_month",
    y="order_count",
    markers=True,
    title="Monthly Order Count Trend"
)
fig.show()

In [43]:
#Delivery Time vs Review Score
fig = px.scatter(
    delivered_df.sample(min(5000, len(delivered_df)), random_state=42),
    x="delivery_time_days",
    y="review_score",
    opacity=0.5,
    title="Delivery Time vs Review Score"
)
fig.show()

In [44]:
#Correlation heatmap
fig = px.imshow(
    corr_df,
    text_auto=True,
    title="Correlation Heatmap"
)
fig.show()

______________________________________
# A/B Testing Preparation
______________________________________

Fast vs Slow Delivery Groups

In [66]:
median_delivery = delivered_df["delivery_time_days"].median()

delivered_df["delivery_group"] = np.where(
    delivered_df["delivery_time_days"] <= median_delivery,
    "Fast",
    "Slow"
)

delivered_df["delivery_group"].value_counts()

,count
delivery_group,
Fast,51839
Slow,43985


# A/B Test 1 | Delivery Speed vs Review Score
H0: Slow and Fast deliveries has the same review score

H1:The median of reviews differentiates

Result:
If p-value < 0.05 → statistically significant difference

In [67]:
fast_reviews = delivered_df.loc[delivered_df["delivery_group"] == "Fast", "review_score"].dropna()
slow_reviews = delivered_df.loc[delivered_df["delivery_group"] == "Slow", "review_score"].dropna()

t_stat_1, p_value_1 = ttest_ind(fast_reviews, slow_reviews, equal_var = False)

print("A/B Test 1 - Delivery Speed vs Review Score")
print("T-statistics:", t_stat_1)
print("P-value:", p_value_1)

A/B Test 1 - Delivery Speed vs Review Score
T-statistics: 58.108197034099405
P-value: 0.0


# A/B Test 2 | Delayed vs On-Time Satisfaction Rate
H0: Review score is the same for delayed and non-delayed orders.

H1: Review scores differentiates

Result:
If p-value < 0.05 → statistically significant difference

In [69]:
ab2_df = delivered_df.dropna(subset = ["is_delayed", "is_satisfied"]).copy()
contingency_ab2 = pd.crosstab(ab2_df["is_delayed"], ab2_df["is_satisfied"])

chi2_2, p_value_2, dof_2, expected_2 = chi2_contingency(contingency_ab2)

print("A/B Test 2 - Delayed vs Satisfaction")
print("Chi-square:", chi2_2)
print("P-value:", p_value_2)
print("\nContingency Table")
print(contingency_ab2)

A/B Test 2 - Delayed vs Satisfaction
Chi-square: 11182.37356041398
P-value: 0.0

Contingency Table
is_satisfied      0      1
is_delayed                
0             15530  73913
1              4675   1706


# A/B Test 3 | Category vs Order Value
H0: The average order value of the selected categories is the same.  

H1: At least one category’s average value is different.

Result:
If p-value < 0.05 → statistically significant difference

In [71]:
top_categories = (
    delivered_df["main_category"]
    .value_counts()
    .head(5)
    .index.tolist()
)

anova_df = delivered_df[delivered_df["main_category"].isin(top_categories)].copy()
groups = [
    anova_df.loc[anova_df["main_category"] == cat, "order_value"].dropna()
    for cat in top_categories
]

anova_stat, anova_p = f_oneway(*groups)

print("A/B Test 3 - Category vs Order Value (ANOVA)")
print("F-statistic:", anova_stat)
print("P-value:", anova_p)
print("Categories used:", top_categories)

A/B Test 3 - Category vs Order Value (ANOVA)
F-statistic: 39.46155963428584
P-value: 4.970862123273817e-33
Categories used: ['cama_mesa_banho', 'beleza_saude', 'esporte_lazer', 'informatica_acessorios', 'moveis_decoracao']


# A/B Test 4 | Credit Card vs Boleto on Satisfaction
H0: The satisfaction rate of customers using credit cards and boleto is the same.

H1: The satisfaction rates are different.

Result:
If p-value < 0.05 → statistically significant difference

In [72]:
ab4_df = delivered_df[delivered_df["main_payment_type"].isin(["credit_card", "boleto"])].copy()

contingency_ab4 = pd.crosstab(ab4_df["main_payment_type"], ab4_df["is_satisfied"])

chi2_4, p_value_4, dof_4, expected_4 = chi2_contingency(contingency_ab4)

print("A/B Test 4 - Credit Card vs Boleto Satisfaction")
print("Chi-square:", chi2_4)
print("P-value:", p_value_4)
print("\nContingency Table")
print(contingency_ab4)

A/B Test 4 - Credit Card vs Boleto Satisfaction
Chi-square: 0.11334730668320159
P-value: 0.7363649892367234

Contingency Table
is_satisfied           0      1
main_payment_type              
boleto              4035  15027
credit_card        15461  57978


__________________________________________
# A/B Test Visualizations
__________________________________________

Test 1 Visual 1 | Fast vs Slow Review Score

In [73]:
ab1_plot = (
    delivered_df.groupby("delivery_group")
    .agg(avg_review_score=("review_score", "mean"),
         order_count=("order_id", "nunique"))
    .reset_index()
)

fig = px.bar(
    ab1_plot,
    x="delivery_group",
    y="avg_review_score",
    text="avg_review_score",
    title="Average Review Score: Fast vs Slow Delivery"
)
fig.show()

Test 2 Visual 2 | Delayed vs On-Time Satisfaction Rate

In [74]:
ab2_plot = (
    ab2_df.groupby("is_delayed")
    .agg(satisfaction_rate=("is_satisfied", "mean"),
         order_count=("order_id", "nunique"))
    .reset_index()
)

ab2_plot["is_delayed"] = ab2_plot["is_delayed"].map({0: "On Time", 1: "Delayed"})

fig = px.bar(
    ab2_plot,
    x="is_delayed",
    y="satisfaction_rate",
    text="satisfaction_rate",
    title="Satisfaction Rate: On-Time vs Delayed Orders"
)
fig.show()

Test 3 Visual 3 | Category vs Order Value

In [75]:
fig = px.box(
    anova_df,
    x="main_category",
    y="order_value",
    title="Order Value Distribution by Top Categories"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

Test 4 Visual 4 | Credit Card vs Boleto Satisfaction Rate

In [76]:
ab4_plot = (
    ab4_df.groupby("main_payment_type")
    .agg(satisfaction_rate=("is_satisfied", "mean"),
         order_count=("order_id", "nunique"))
    .reset_index()
)

fig = px.bar(
    ab4_plot,
    x="main_payment_type",
    y="satisfaction_rate",
    text="satisfaction_rate",
    title="Satisfaction Rate by Payment Type: Credit Card vs Boleto"
)
fig.show()

_________________________
# Final Test Summary
_____________________

In [77]:
test_summary = pd.DataFrame({
    "test_name": [
        "Fast vs Slow Delivery Review Score",
        "Delayed vs On-Time Satisfaction",
        "Category vs Order Value (ANOVA)",
        "Credit Card vs Boleto Satisfaction"
    ],
    "test_type": [
        "Independent T-Test",
        "Chi-Square",
        "ANOVA",
        "Chi-Square"
    ],
    "p_value": [
        p_value_1,
        p_value_2,
        anova_p,
        p_value_4
    ]
})

test_summary["significant_at_0_05"] = np.where(test_summary["p_value"] < 0.05, "Yes", "No")
test_summary

,test_name,test_type,p_value,significant_at_0_05
0,Fast vs Slow Delivery Review Score,Independent T-Test,0.000000e+00,Yes
1,Delayed vs On-Time Satisfaction,Chi-Square,0.000000e+00,Yes
2,Category vs Order Value (ANOVA),ANOVA,4.970862e-33,Yes
3,Credit Card vs Boleto Satisfaction,Chi-Square,7.363650e-01,No


## Key Findings

- Delivery delays significantly reduce customer satisfaction
- Order value varies significantly across product categories
- Payment method shows limited impact on satisfaction

## Limitations

- dataset limited to 2016–2018
- some missing values in reviews and delivery dates

## Next Steps

- customer segmentation
- predictive modeling
- retention analysis